# CSE480 Jigsaw Reconstruction — Demonstration

From-scratch operators (enhancement, Canny, CCL), then board-size inference and the quality score `Q`.

**Q formula (code, report, and this notebook):** `Q = 0.5·position + 0.3·orientation + 0.2·edge`. Missing terms are 0, so without pose GT, `Q ≤ 0.2`. Lead with identity-neighbour accuracy, not Q.

Full reconstruction: `python main.py reconstruct --method classical --config configs/classical.yaml --input <image>`.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from src.core.convolution import ConvolutionEngine
from src.core.grid import infer_board_shape
from src.edge_detection import CannyEdgeDetector, SobelOperator
from src.enhancement import HistogramEqualizer, MeanFilter, MedianFilter, UnsharpMask
from src.evaluation import canonical_Q
from src.segmentation import ConnectedComponentLabeler, orient_foreground
from src.thresholding import OtsuThreshold

img = np.zeros((64, 64), dtype=np.float64)
img[16:48, 16:32] = 200
img[16:48, 36:52] = 200
img[20, 20] = 0  # impulse

smooth = MedianFilter(3).apply(MeanFilter(3, ConvolutionEngine()).apply(img))
sharp = UnsharpMask(3, 1.0, 0.8).apply(smooth)
eq = HistogramEqualizer().apply(sharp)
print("mean/median/unsharp/equalize shapes", smooth.shape, sharp.shape, eq.shape)

binary = orient_foreground(eq, OtsuThreshold().threshold(eq))
labels = ConnectedComponentLabeler(min_area=20).label(binary)
print("CCL labels", sorted(set(labels.ravel()) - {0}))
print("board for 9/16/35 pieces", infer_board_shape(9), infer_board_shape(16), infer_board_shape(35))

edge = CannyEdgeDetector(gradient=SobelOperator(), t_low=20, t_high=50).detect(img)
print("Canny edge pixels", int((edge.edges > 0).sum()))
print("Q = 0.5 pos + 0.3 ori + 0.2 edge; missing terms are 0")
print("example without pose GT, neighbour=0.20 -> Q", round(canonical_Q(None, None, 0.20), 4))

mean/median/unsharp/equalize shapes (64, 64) (64, 64) (64, 64)
CCL labels [np.int32(1), np.int32(2)]
board for 9/16/35 pieces (3, 3) (4, 4) (7, 5)
Canny edge pixels 230
Q = 0.5 pos + 0.3 ori + 0.2 edge; missing terms are 0
example without pose GT, neighbour=0.20 -> Q 0.04
